# DATASCAPE 2026 — pseudo-label policy + ConvNeXt-Small A/B (A100, 384 px)

Notebook ini membangun policy pseudo-label v1 secara deterministik dari dua annotator OOF, lalu melatih dua kontrol yang identik pada development split:

- **A — baseline:** label asli, weight 1;
- **B — candidate:** 77 flip T1 dan 123 sampel T2 dengan weight 0.5.

Keduanya memakai `convnext_small.fb_in22k_ft_in1k_384`, input 384, seed/augmentasi/epoch/sampler sama, dan BF16 AMP pada NVIDIA A100. Holdout baru dibuka setelah kedua training selesai. Keputusan `Lainnya` kandidat dimask dengan gate baseline sehingga identik secara deterministik.

Prasyarat Drive dari run sebelumnya:

- `MyDrive/DATASCAPE/siglip2_first_pipeline/siglip2_grouped_manifest.csv`
- `MyDrive/DATASCAPE/convnext_grouped_oof_teacher_seed42/convnext_siglip_pseudo_label_audit.csv`

Tambahkan `KAGGLE_API_TOKEN` ke Colab Secrets jika dataset belum tersedia di `/content`. Setelah itu pilih **A100 GPU** dan jalankan **Run all**.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'torch': 'torch', 'torchvision': 'torchvision', 'timm': 'timm>=1.0.20',
    'sklearn': 'scikit-learn', 'scipy': 'scipy', 'tqdm': 'tqdm',
    'kagglehub': 'kagglehub',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import contextlib
import gc
import hashlib
import json
import math
import os
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from timm.utils import ModelEmaV3
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    raise RuntimeError('Notebook ini dikonfigurasi untuk Google Colab T4.')

from google.colab import drive
drive.mount('/content/drive')
print('Python:', sys.version.split()[0], '| torch:', torch.__version__, '| timm:', timm.__version__)

In [ ]:
# Fixed, preregistered experiment contract.
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
COMPETITION = 'datascape-2026-data-mining-competition'
MODEL_ID = 'hf_hub:timm/convnext_small.fb_in22k_ft_in1k_384'
IMAGE_SIZE = 384
FIXED_EPOCHS = 12
TARGET_EFFECTIVE_BATCH = 128
BATCH_CANDIDATES = (128, 64, 32, 16, 8)
MAX_MEMORY_FRACTION = 0.90
REFERENCE_BATCH = 128
REFERENCE_LR = 2e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 1
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.08
BALANCE_STRENGTH = 0.50
GRAD_CLIP_NORM = 1.0
DROP_RATE = 0.20
DROP_PATH_RATE = 0.20
MODEL_EMA_DECAY = 0.9997
USE_HORIZONTAL_FLIP_TTA = True
BOOTSTRAP_REPLICATES = 2000

T1_THRESHOLD = {'Aman': 2.0, 'Rusak': 3.5}
T2_THRESHOLD = {'Aman': 1.0, 'Rusak': 1.5}
EXPECTED_MANIFEST_SHA256 = '34d911482f95de94c4a854cac927b5ba5d7aa3ba6e55973f7d1201825563a7d2'
EXPECTED_AUDIT_SHA256 = '399e7ae3847a105b13211610a53ab60e0ef7ab0f38d7281435ef2ef6fb24be3c'

DRIVE_ROOT = Path('/content/drive/MyDrive/DATASCAPE')
MANIFEST_PATH = DRIVE_ROOT / 'siglip2_first_pipeline/siglip2_grouped_manifest.csv'
AUDIT_PATH = DRIVE_ROOT / 'convnext_grouped_oof_teacher_seed42/convnext_siglip_pseudo_label_audit.csv'
OUTPUT_DIR = DRIVE_ROOT / 'pseudo_label_small_384_a100_e12_ablation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_ROOT = Path('/content') / COMPETITION
DOWNLOAD_ROOT = Path('/content/datascape_dataset')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak aktif. Pilih Runtime > Change runtime type > T4 GPU.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in GPU_NAME.upper():
    raise RuntimeError(f'Runtime guard: expected NVIDIA A100, got {GPU_NAME}.')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
AMP_DTYPE = torch.bfloat16
USE_GRAD_SCALER = AMP_DTYPE == torch.float16
AMP_DESCRIPTION = 'bfloat16_no_grad_scaler'
print('Runtime:', GPU_NAME, f'| VRAM={GPU_VRAM_GIB:.1f} GiB | AMP=BF16 | image={IMAGE_SIZE}')
print('Output:', OUTPUT_DIR)

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(zip_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_dataset_root(start):
    start = Path(start)
    candidates = [start]
    if start.exists():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Train_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(start)


def resolve_dataset_root():
    for candidate in (LOCAL_DATA_ROOT, DOWNLOAD_ROOT):
        try:
            return find_dataset_root(candidate)
        except FileNotFoundError:
            pass
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    try:
        from google.colab import userdata
        token = (userdata.get('KAGGLE_API_TOKEN') or token).strip()
    except Exception:
        pass
    if not token:
        raise RuntimeError('Dataset belum ada dan KAGGLE_API_TOKEN tidak ditemukan di Colab Secrets.')
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(token)
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, DOWNLOAD_ROOT))
    return find_dataset_root(downloaded)

for required_path in (MANIFEST_PATH, AUDIT_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing prerequisite: {required_path}')
if sha256_file(MANIFEST_PATH) != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Grouped manifest hash mismatch; do not run an A/B on a different split.')
if sha256_file(AUDIT_PATH) != EXPECTED_AUDIT_SHA256:
    raise RuntimeError('Teacher audit hash mismatch; expected the completed seed-42 OOF artifact.')

DATA_ROOT = resolve_dataset_root()
manifest = pd.read_csv(MANIFEST_PATH)
audit = pd.read_csv(AUDIT_PATH)
if len(manifest) != 12224 or len(audit) != 11043:
    raise ValueError(f'Unexpected rows: manifest={len(manifest)}, audit={len(audit)}')
if manifest.groupby('dedup_group').partition.nunique().max() != 1:
    raise ValueError('A duplicate group crosses development/holdout.')
if set(audit.filepath) != set(manifest.loc[manifest.partition.eq('development'), 'filepath']):
    raise ValueError('Teacher audit does not align with development rows.')
print('Dataset:', DATA_ROOT)
print('Manifest:', len(manifest), '| development:', len(audit), '| holdout:', int(manifest.partition.eq('holdout').sum()))

In [ ]:
# Build the frozen v1 policy. The score is a prior-adjusted ranking score,
# not a calibrated likelihood ratio: ConvNeXt used weighted sampling and both
# annotators used label smoothing.
group_size = manifest.groupby('dedup_group').size()
groups_with_lainnya = set(manifest.loc[manifest.label.eq('Lainnya'), 'dedup_group'])
audit['group_size'] = audit.dedup_group.map(group_size).astype(int)
audit['component_contains_lainnya'] = audit.dedup_group.isin(groups_with_lainnya)
priors = audit.label_original.value_counts().reindex(CLASSES) / len(audit)

for model in ('convnext', 'siglip'):
    target_probability = np.full(len(audit), np.nan, dtype=np.float64)
    original_probability = np.empty(len(audit), dtype=np.float64)
    prior_offset = np.full(len(audit), np.nan, dtype=np.float64)
    for label in CLASSES:
        original_mask = audit.label_original.eq(label).to_numpy()
        target_mask = audit.candidate_target.eq(label).to_numpy()
        original_probability[original_mask] = audit.loc[original_mask, f'p_{label}_{model}']
        target_probability[target_mask] = audit.loc[target_mask, f'p_{label}_{model}']
    valid_target = audit.candidate_target.isin(CLASSES).to_numpy()
    prior_offset[valid_target] = [
        np.log(priors[target] / priors[original])
        for target, original in zip(
            audit.loc[valid_target, 'candidate_target'],
            audit.loc[valid_target, 'label_original'],
        )
    ]
    audit[f'loglr_{model}'] = (
        np.log(np.maximum(target_probability, 1e-8) / np.maximum(original_probability, 1e-8))
        - prior_offset
    )
audit['min_prior_adjusted_log_odds'] = audit[
    ['loglr_convnext', 'loglr_siglip']
].min(axis=1)

strict = audit.groupby('dedup_group').agg(
    all_models_agree=('cross_model_agreement', 'all'),
    consensus_targets=('candidate_target', 'nunique'),
    nonempty_targets=('candidate_target', lambda values: values.notna().all() and values.astype(str).ne('').all()),
)
strict['strict_group_consensus'] = (
    strict.all_models_agree & strict.consensus_targets.eq(1) & strict.nonempty_targets
)
audit['strict_group_consensus'] = audit.dedup_group.map(strict.strict_group_consensus).fillna(False)
audit['pseudo_gate'] = (
    audit.label_original.isin(('Aman', 'Rusak'))
    & audit.candidate_target.isin(('Aman', 'Rusak'))
    & audit.candidate_target.ne(audit.label_original)
    & audit.partition_unanimous.astype(bool)
    & audit.p_Lainnya_convnext.lt(0.30)
    & audit.p_Lainnya_siglip.lt(0.30)
    & audit.group_size.le(8)
    & ~audit.component_contains_lainnya
    & audit.strict_group_consensus
)

audit['raw_tier'] = 'T3'
for original in ('Aman', 'Rusak'):
    direction = audit.pseudo_gate & audit.label_original.eq(original)
    audit.loc[direction & audit.min_prior_adjusted_log_odds.ge(T2_THRESHOLD[original]), 'raw_tier'] = 'T2'
    audit.loc[direction & audit.min_prior_adjusted_log_odds.ge(T1_THRESHOLD[original]), 'raw_tier'] = 'T1'

raw_t1 = audit.loc[audit.raw_tier.eq('T1')]
t1_group_targets = raw_t1.groupby('dedup_group').candidate_target.agg(
    lambda values: values.iloc[0] if values.nunique() == 1 else ''
)
if t1_group_targets.eq('').any():
    raise RuntimeError('A T1 group has conflicting targets.')

audit['effective_label'] = audit.label_original
audit['loss_weight'] = 1.0
audit['policy_tier'] = 'T3'
audit['group_propagated'] = False
in_t1_group = audit.dedup_group.isin(t1_group_targets.index)
propagated_target = audit.dedup_group.map(t1_group_targets)
flipped = in_t1_group & audit.label_original.ne(propagated_target)
audit.loc[flipped, 'effective_label'] = propagated_target[flipped]
audit.loc[flipped, 'policy_tier'] = 'T1'
audit.loc[flipped & ~audit.raw_tier.eq('T1'), 'group_propagated'] = True

t2 = audit.raw_tier.eq('T2') & ~in_t1_group
audit.loc[t2, 'policy_tier'] = 'T2'
audit.loc[t2, 'loss_weight'] = 0.5

changed = audit.effective_label.ne(audit.label_original)
assert int(changed.sum()) == 77
assert int((changed & audit.group_propagated).sum()) == 4
assert int(t2.sum()) == 123
assert not audit.loc[audit.label_original.eq('Lainnya'), 'effective_label'].ne('Lainnya').any()
assert not audit.loc[audit.policy_tier.ne('T3'), 'group_size'].gt(8).any()

policy_columns = [
    'filepath', 'label_original', 'effective_label', 'loss_weight', 'policy_tier',
    'group_propagated', 'dedup_group', 'group_size', 'candidate_target',
    'min_prior_adjusted_log_odds', 'loglr_convnext', 'loglr_siglip',
]
policy_path = OUTPUT_DIR / 'pseudo_label_policy_v1.csv'
audit[policy_columns].to_csv(policy_path, index=False)
policy_sha256 = sha256_file(policy_path)
policy_summary = {
    'protocol': 'two_oof_annotators_empirical_prior_score_group_propagation_v1',
    'manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'audit_sha256': EXPECTED_AUDIT_SHA256,
    'policy_sha256': policy_sha256,
    'gate_rows': int(audit.pseudo_gate.sum()),
    'raw_t1_rows': int(audit.raw_tier.eq('T1').sum()),
    'flipped_rows': int(changed.sum()),
    'group_propagated_additional_flips': int((changed & audit.group_propagated).sum()),
    'downweighted_rows': int(t2.sum()),
    'flip_directions': {
        f'{original}_to_{target}': int(count)
        for (original, target), count in audit.loc[changed].groupby(
            ['label_original', 'effective_label']
        ).size().items()
    },
}
(OUTPUT_DIR / 'pseudo_label_policy_v1_summary.json').write_text(json.dumps(policy_summary, indent=2) + '\n')
print(json.dumps(policy_summary, indent=2))

In [ ]:
# Seal the holdout labels until both models have finished training.
development = manifest.loc[manifest.partition.eq('development'), [
    'filepath', 'label', 'dedup_group'
]].rename(columns={'label': 'label_original'})
development['image_path'] = development.filepath.map(lambda value: str(DATA_ROOT / value))
if not development.image_path.map(lambda value: Path(value).is_file()).all():
    raise FileNotFoundError('Some development images are missing.')
policy = audit[[
    'filepath', 'effective_label', 'loss_weight', 'policy_tier', 'group_propagated'
]]
development = development.merge(policy, on='filepath', validate='one_to_one')
if not development.label_original.eq(audit.set_index('filepath').loc[development.filepath, 'label_original'].to_numpy()).all():
    raise ValueError('Policy/original-label alignment failed.')

baseline_frame = development.copy()
baseline_frame['train_label'] = baseline_frame.label_original
baseline_frame['sample_loss_weight'] = 1.0
candidate_frame = development.copy()
candidate_frame['train_label'] = candidate_frame.effective_label
candidate_frame['sample_loss_weight'] = candidate_frame.loss_weight.astype(float)

holdout_index = manifest.loc[manifest.partition.eq('holdout'), ['filepath', 'dedup_group']].copy()
holdout_index['image_path'] = holdout_index.filepath.map(lambda value: str(DATA_ROOT / value))
assert len(holdout_index) == 1181
assert set(baseline_frame.filepath).isdisjoint(holdout_index.filepath)

# Remove the live manifest and its holdout labels before training.
del manifest
print('Training rows:', len(development), '| sealed holdout rows:', len(holdout_index))
print('Candidate effective-label counts:', candidate_frame.train_label.value_counts().to_dict())

In [ ]:
fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)

class Letterbox:
    def __init__(self, size):
        self.size = int(size)
    def __call__(self, image):
        return ImageOps.pad(
            image, (self.size, self.size), Image.Resampling.BILINEAR,
            color=fill, centering=(0.5, 0.5),
        )


def load_rgb_image(path):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            rgba_fill = tuple(int(value * 255) for value in IMAGENET_MEAN) + (255,)
            image = Image.alpha_composite(Image.new('RGBA', rgba.size, rgba_fill), rgba).convert('RGB')
        else:
            image = image.convert('RGB')
        return image.copy()

TRAIN_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomAffine(
        degrees=3.0, translate=(0.02, 0.02), scale=(0.97, 1.03),
        interpolation=transforms.InterpolationMode.BILINEAR, fill=fill,
    ),
    transforms.ColorJitter(brightness=0.06, contrast=0.06, saturation=0.06, hue=0.01),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
EVAL_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class TrainDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = TRAIN_TRANSFORM(load_rgb_image(row.image_path))
        return (
            image,
            CLASS_TO_INDEX[row.train_label],
            float(row.sample_loss_weight),
            index,
        )

class EvalDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return EVAL_TRANSFORM(load_rgb_image(row.image_path)), index

NUM_WORKERS = min(4, os.cpu_count() or 2)
PIN_MEMORY = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(frame, batch_size, seed):
    counts = frame.train_label.value_counts()
    sampling_weights = frame.train_label.map(
        lambda label: counts[label] ** -BALANCE_STRENGTH
    ).to_numpy()
    sampler_generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.tensor(sampling_weights, dtype=torch.double), len(frame),
        replacement=True, generator=sampler_generator,
    )
    loader_generator = torch.Generator().manual_seed(seed + 1)
    return DataLoader(
        TrainDataset(frame), batch_size=batch_size, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=2 if NUM_WORKERS > 0 else None,
        worker_init_fn=seed_worker, generator=loader_generator,
    )


def make_eval_loader(frame, batch_size):
    return DataLoader(
        EvalDataset(frame), batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

print('Loader workers:', NUM_WORKERS, '| preview:', TrainDataset(baseline_frame)[0][0].shape)

In [ ]:
def autocast_context():
    return torch.autocast('cuda', dtype=AMP_DTYPE)


def build_model():
    model = timm.create_model(
        MODEL_ID, pretrained=True, num_classes=len(CLASSES),
        drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE,
    ).cuda(memory_format=torch.channels_last)
    return model


def probe_batch_size():
    for batch_size in BATCH_CANDIDATES:
        model = ema = optimizer = scaler = images = targets = None
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            model = build_model()
            ema = ModelEmaV3(
                model, decay=MODEL_EMA_DECAY, use_warmup=True,
                warmup_gamma=1.0, warmup_power=2/3,
            )
            optimizer = torch.optim.AdamW(model.parameters(), lr=REFERENCE_LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
            images = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE, device='cuda').contiguous(
                memory_format=torch.channels_last
            )
            targets = torch.randint(0, len(CLASSES), (batch_size,), device='cuda')
            with autocast_context():
                loss = F.cross_entropy(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            fraction = torch.cuda.max_memory_reserved() / torch.cuda.get_device_properties(0).total_memory
            print(f'batch {batch_size}: peak reserved {fraction:.1%}')
            if fraction <= MAX_MEMORY_FRACTION:
                return batch_size
        except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
            if 'out of memory' not in str(error).lower():
                raise
            print('OOM at batch', batch_size)
        finally:
            del model, ema, optimizer, scaler, images, targets
            gc.collect()
            torch.cuda.empty_cache()
    raise RuntimeError('ConvNeXt-Small 384 does not fit even at batch 8.')

BATCH_SIZE = probe_batch_size()
GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH / BATCH_SIZE))
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = min(64, BATCH_SIZE * 2)
BASE_LR = REFERENCE_LR * math.sqrt(EFFECTIVE_BATCH_SIZE / REFERENCE_BATCH)
print({
    'physical_batch': BATCH_SIZE,
    'gradient_accumulation': GRAD_ACCUM_STEPS,
    'effective_batch': EFFECTIVE_BATCH_SIZE,
    'eval_batch': EVAL_BATCH_SIZE,
    'base_lr': BASE_LR,
    'image_size': IMAGE_SIZE,
})

In [ ]:
def set_run_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def learning_rate_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(FIXED_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


def checkpoint_contract(condition):
    return {
        'condition': condition,
        'model_id': MODEL_ID,
        'image_size': IMAGE_SIZE,
        'seed': SEED,
        'fixed_epochs': FIXED_EPOCHS,
        'manifest_sha256': EXPECTED_MANIFEST_SHA256,
        'audit_sha256': EXPECTED_AUDIT_SHA256,
        'policy_sha256': policy_sha256,
        'physical_batch': BATCH_SIZE,
        'effective_batch': EFFECTIVE_BATCH_SIZE,
    }


def train_condition(condition, frame):
    condition_dir = OUTPUT_DIR / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = condition_dir / 'final_ema.pt'
    complete_path = condition_dir / 'complete.json'
    contract = checkpoint_contract(condition)
    if checkpoint_path.is_file() and complete_path.is_file():
        saved_contract = json.loads(complete_path.read_text())
        if saved_contract != contract:
            raise RuntimeError(f'{condition} checkpoint contract mismatch: {saved_contract}')
        print('Resume:', condition, checkpoint_path)
        return

    set_run_seed(SEED)
    loader = make_train_loader(frame, BATCH_SIZE, SEED)
    model = build_model()
    ema = ModelEmaV3(model, decay=MODEL_EMA_DECAY, use_warmup=True, warmup_gamma=1.0, warmup_power=2/3)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
    history = []
    optimizer_updates = 0

    for epoch in range(FIXED_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = total_weight = 0.0
        started = time.time()
        number_of_batches = len(loader)
        progress = tqdm(loader, desc=f'{condition} epoch {epoch + 1}/{FIXED_EPOCHS}')
        for step, (images, targets, loss_weights, _) in enumerate(progress):
            images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
            targets = targets.cuda(non_blocking=True)
            loss_weights = loss_weights.cuda(non_blocking=True)
            window_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            window_size = min(GRAD_ACCUM_STEPS, number_of_batches - window_start)
            with autocast_context():
                per_sample = F.cross_entropy(
                    model(images), targets, reduction='none', label_smoothing=LABEL_SMOOTHING
                )
                raw_loss = (per_sample * loss_weights).sum() / loss_weights.sum().clamp_min(1e-6)
                loss = raw_loss / window_size
            scaler.scale(loss).backward()
            should_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == number_of_batches
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                optimizer_updates += 1
                ema.update(model, step=optimizer_updates)
            total_loss += float((per_sample * loss_weights).sum().item())
            total_weight += float(loss_weights.sum().item())
        row = {
            'condition': condition, 'epoch': epoch + 1, 'lr': lr,
            'weighted_train_loss': total_loss / total_weight,
            'seconds': time.time() - started,
        }
        history.append(row)
        print(row)
        pd.DataFrame(history).to_csv(condition_dir / 'history.csv', index=False)

    temporary = checkpoint_path.with_suffix('.pt.tmp')
    torch.save({'model_state': {k: v.cpu() for k, v in ema.module.state_dict().items()}}, temporary)
    temporary.replace(checkpoint_path)
    temporary_contract = complete_path.with_suffix('.json.tmp')
    temporary_contract.write_text(json.dumps(contract, indent=2) + '\n')
    temporary_contract.replace(complete_path)
    del model, ema, optimizer, scaler, loader
    gc.collect()
    torch.cuda.empty_cache()
    print('Completed:', condition)

train_condition('baseline_original', baseline_frame)
train_condition('candidate_policy_v1', candidate_frame)

In [ ]:
# Both recipes are now frozen. Open holdout labels once for paired go/no-go evaluation.
evaluation_manifest = pd.read_csv(MANIFEST_PATH)
holdout = holdout_index.merge(
    evaluation_manifest.loc[evaluation_manifest.partition.eq('holdout'), ['filepath', 'label']],
    on='filepath', validate='one_to_one',
).rename(columns={'label': 'label_original'})
del evaluation_manifest
assert len(holdout) == 1181
assert holdout.label_original.value_counts().to_dict() == {'Aman': 903, 'Rusak': 244, 'Lainnya': 34}

@torch.inference_mode()
def predict_checkpoint(condition, frame):
    checkpoint = torch.load(
        OUTPUT_DIR / condition / 'final_ema.pt', map_location='cpu', weights_only=True
    )
    model = build_model()
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    loader = make_eval_loader(frame, EVAL_BATCH_SIZE)
    chunks = []
    for images, _ in tqdm(loader, desc=f'holdout {condition}'):
        images = images.cuda(non_blocking=True).contiguous(memory_format=torch.channels_last)
        with autocast_context():
            probabilities = model(images).softmax(1)
            if USE_HORIZONTAL_FLIP_TTA:
                probabilities = 0.5 * (
                    probabilities + model(torch.flip(images, dims=[3])).softmax(1)
                )
        chunks.append(probabilities.float().cpu().numpy())
    result = np.concatenate(chunks)
    del model, loader, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    return result

baseline_probabilities = predict_checkpoint('baseline_original', holdout)
candidate_probabilities = predict_checkpoint('candidate_policy_v1', holdout)
class_array = np.asarray(CLASSES)
baseline_prediction = class_array[baseline_probabilities.argmax(1)]
candidate_ar_prediction = np.asarray(('Aman', 'Rusak'))[
    candidate_probabilities[:, :2].argmax(1)
]
# Structural frozen gate: candidate can only decide Aman versus Rusak outside
# the baseline Lainnya region.
candidate_masked_prediction = np.where(
    baseline_prediction == 'Lainnya', 'Lainnya', candidate_ar_prediction
)
assert np.array_equal(
    baseline_prediction == 'Lainnya', candidate_masked_prediction == 'Lainnya'
)
print('Frozen Lainnya decisions:', int((baseline_prediction == 'Lainnya').sum()))

In [ ]:
def class_metric_table(y_true, y_pred, condition):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    return pd.DataFrame({
        'condition': condition, 'label': CLASSES, 'precision': precision,
        'recall': recall, 'f1': f1, 'support': support,
    })

truth = holdout.label_original.to_numpy()
metrics = pd.concat([
    class_metric_table(truth, baseline_prediction, 'baseline_original'),
    class_metric_table(truth, candidate_masked_prediction, 'candidate_policy_v1_masked'),
], ignore_index=True)
print(metrics.to_string(index=False))

for name, prediction in (
    ('baseline_original', baseline_prediction),
    ('candidate_policy_v1_masked', candidate_masked_prediction),
):
    matrix = confusion_matrix(truth, prediction, labels=CLASSES)
    pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).rename_axis('label_original').to_csv(
        OUTPUT_DIR / f'{name}_holdout_confusion.csv'
    )
    print('\n', name, '\n', matrix)

baseline_correct = baseline_prediction == truth
candidate_correct = candidate_masked_prediction == truth
gain = int((~baseline_correct & candidate_correct).sum())
harm = int((baseline_correct & ~candidate_correct).sum())
mcnemar_p = float(binomtest(min(gain, harm), gain + harm, 0.5).pvalue) if gain + harm else 1.0
print({'gain': gain, 'harm': harm, 'mcnemar_exact_p': mcnemar_p})

# Paired cluster bootstrap over duplicate groups.
rng = np.random.default_rng(2607)
unique_groups = holdout.dedup_group.unique()
group_indices = {
    group: np.flatnonzero(holdout.dedup_group.to_numpy() == group)
    for group in unique_groups
}
bootstrap_rows = []
for replicate in tqdm(range(BOOTSTRAP_REPLICATES), desc='cluster bootstrap'):
    sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
    indices = np.concatenate([group_indices[group] for group in sampled_groups])
    y = truth[indices]
    base = baseline_prediction[indices]
    candidate = candidate_masked_prediction[indices]
    base_f1 = f1_score(y, base, labels=CLASSES, average=None, zero_division=0)
    candidate_f1 = f1_score(y, candidate, labels=CLASSES, average=None, zero_division=0)
    bootstrap_rows.append({
        'replicate': replicate,
        'delta_macro_f1': float(candidate_f1.mean() - base_f1.mean()),
        'delta_rusak_f1': float(candidate_f1[CLASS_TO_INDEX['Rusak']] - base_f1[CLASS_TO_INDEX['Rusak']]),
    })
bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(OUTPUT_DIR / 'paired_cluster_bootstrap.csv', index=False)

base_f1 = metrics.loc[metrics.condition.eq('baseline_original')].set_index('label').f1
candidate_f1 = metrics.loc[metrics.condition.eq('candidate_policy_v1_masked')].set_index('label').f1
delta_rusak = float(candidate_f1.Rusak - base_f1.Rusak)
delta_macro = float(candidate_f1.mean() - base_f1.mean())
rusak_ci = [float(value) for value in bootstrap.delta_rusak_f1.quantile([0.025, 0.975])]
macro_ci = [float(value) for value in bootstrap.delta_macro_f1.quantile([0.025, 0.975])]
accepted = bool(delta_rusak >= 0.01 and rusak_ci[0] > 0)

predictions = holdout[['filepath', 'label_original', 'dedup_group']].copy()
predictions['baseline_prediction'] = baseline_prediction
predictions['candidate_prediction_masked'] = candidate_masked_prediction
for index, label in enumerate(CLASSES):
    predictions[f'p_{label}_baseline'] = baseline_probabilities[:, index]
    predictions[f'p_{label}_candidate'] = candidate_probabilities[:, index]
predictions.to_csv(OUTPUT_DIR / 'paired_holdout_predictions.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'paired_holdout_class_metrics.csv', index=False)

summary = {
    'protocol': 'convnext_small_384_a100_e12_baseline_vs_policy_v1_paired_holdout',
    'model_id': MODEL_ID,
    'image_size': IMAGE_SIZE,
    'gpu': GPU_NAME,
    'amp': AMP_DESCRIPTION,
    'physical_batch': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'effective_batch': EFFECTIVE_BATCH_SIZE,
    'fixed_epochs': FIXED_EPOCHS,
    'policy_sha256': policy_sha256,
    'holdout_rows': int(len(holdout)),
    'baseline_macro_f1': float(base_f1.mean()),
    'candidate_macro_f1': float(candidate_f1.mean()),
    'delta_macro_f1': delta_macro,
    'delta_macro_f1_cluster_bootstrap_95ci': macro_ci,
    'baseline_rusak_f1': float(base_f1.Rusak),
    'candidate_rusak_f1': float(candidate_f1.Rusak),
    'delta_rusak_f1': delta_rusak,
    'delta_rusak_f1_cluster_bootstrap_95ci': rusak_ci,
    'gain': gain,
    'harm': harm,
    'mcnemar_exact_p': mcnemar_p,
    'lainnya_decisions_identical': True,
    'accepted': accepted,
    'acceptance_rule': 'delta Rusak F1 >= 0.01 and paired cluster-bootstrap lower bound > 0',
}
(OUTPUT_DIR / 'paired_holdout_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

In [ ]:
# Bundle only compact evaluation outputs for download. Checkpoints remain in Drive.
bundle_path = Path('/content/pseudo_label_small_384_t4_results.zip')
result_names = [
    'pseudo_label_policy_v1.csv', 'pseudo_label_policy_v1_summary.json',
    'baseline_original_holdout_confusion.csv',
    'candidate_policy_v1_masked_holdout_confusion.csv',
    'paired_holdout_predictions.csv', 'paired_holdout_class_metrics.csv',
    'paired_cluster_bootstrap.csv', 'paired_holdout_summary.json',
]
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in result_names:
        archive.write(OUTPUT_DIR / name, arcname=name)
print('Result bundle:', bundle_path, '| bytes:', bundle_path.stat().st_size)
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception as error:
    print('Automatic download unavailable:', error)